# Recorrido reproducible del trabajo final

**Investigación Operativa II — UCASAL, 2026.**
Pregunta del estudio: ¿cuántos preparadores de pedidos online conviene asignar para cumplir un tiempo objetivo de preparación bajo demanda baja, media y alta?

Este notebook tiene dos partes:

1. **Reproducción (secciones 1–7).** Vuelve a ejecutar el pipeline completo (`scripts/01`–`05`) con `config/experimento.yaml` y comprueba que los resultados numéricos coinciden con los entregados. Después muestra, para cada etapa, las tablas y figuras de las que salen los números del informe.
2. **Exploración (sección 8).** Permite cambiar el plazo, el cumplimiento exigido, la demanda o el tiempo de servicio y volver a simular. **Esos resultados no son los del informe**: sirven para entender el modelo, no reemplazan a los oficiales.

Toda la lógica está en el paquete `src/pedidos/`; el notebook solo la llama y muestra los resultados.

**Cómo ejecutarlo:** desde la raíz del proyecto, `pip install -r requirements.txt` y luego `jupyter notebook notebooks/recorrido.ipynb`; en el menú, *Run → Run All Cells*.
La reproducción completa tarda alrededor de dos minutos.

## 1. Entorno y configuración

In [ ]:
import copy
import hashlib
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

# Raíz del proyecto: funciona tanto si Jupyter se abrió en la raíz como en notebooks/.
RAIZ = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
assert (RAIZ / "config" / "experimento.yaml").exists(), f"No se encontró config/ desde {Path.cwd()}"

# Si el paquete no se instaló con `pip install -e .`, se usa directamente desde src/.
try:
    import pedidos  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(RAIZ / "src"))

from pedidos import experimentos as exp
from pedidos.colas import erlang_c
from pedidos.verificacion import verificar_mmc

CONFIG = yaml.safe_load((RAIZ / "config" / "experimento.yaml").read_text(encoding="utf-8"))
RESULTADOS = RAIZ / "resultados"
pd.set_option("display.float_format", "{:.4f}".format)
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · raíz: {RAIZ}")

## 2. Reproducir el pipeline

La celda siguiente registra la huella (SHA-256) de los resultados numéricos entregados, ejecuta los cinco scripts en orden y vuelve a calcular las huellas.
Se comparan los CSV de `resultados/` y las tablas de `informe/tablas/`, que contienen todos los números citados en el informe.
Las figuras PDF quedan fuera de la comparación porque sus bytes dependen de la versión de Matplotlib, aunque los datos graficados sean los mismos.

Toda la aleatoriedad usa semillas fijadas en la configuración, así que la misma configuración debe producir archivos idénticos.
Con `CORRER_PIPELINE = False` se omite la ejecución y el resto del notebook usa los resultados ya presentes.

In [ ]:
CORRER_PIPELINE = True

COMPARADOS = sorted(RESULTADOS.glob("*.csv")) + sorted((RAIZ / "informe" / "tablas").glob("*.tex"))


def huellas(rutas):
    return {r.relative_to(RAIZ).as_posix(): hashlib.sha256(r.read_bytes()).hexdigest() for r in rutas if r.exists()}


antes = huellas(COMPARADOS)
if CORRER_PIPELINE:
    for script in sorted((RAIZ / "scripts").glob("0*_*.py")):
        inicio = time.perf_counter()
        corrida = subprocess.run([sys.executable, str(script)], cwd=RAIZ, capture_output=True, text=True)
        if corrida.returncode != 0:
            raise RuntimeError(f"{script.name} falló:\n{corrida.stderr}")
        print(f"{script.name:28s} ok  ({time.perf_counter() - inicio:5.1f} s)")

despues = huellas(sorted(RESULTADOS.glob("*.csv")) + sorted((RAIZ / "informe" / "tablas").glob("*.tex")))
comparacion = pd.DataFrame(
    {
        "archivo": sorted(despues),
        "estado": [
            "nuevo" if a not in antes else ("idéntico" if antes[a] == despues[a] else "DISTINTO")
            for a in sorted(despues)
        ],
    }
)
print(comparacion["estado"].value_counts().to_string())
comparacion

## 3. Datos: histórico sintético

El estudio usa un histórico sintético de 26 semanas (Poisson no homogéneo, ~200 pedidos/día, 8 a 21 h), calibrado con el calendario semanal del ERP.
La justificación está en `docs/inspeccion-erp.md`.
La tabla compara indicadores del ERP con los del histórico sintético; el gráfico muestra la demanda media por hora y por día de la semana.

In [ ]:
resumen_datos = pd.read_csv(RESULTADOS / "datos_resumen.csv", index_col="indicador")
display(resumen_datos)

serie = pd.read_csv(RAIZ / "datos" / "procesados" / "serie_horaria.csv", index_col="hora", parse_dates=["hora"])["pedidos"]
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
perfil = serie.groupby([serie.index.day_of_week, serie.index.hour]).mean().unstack(0)
perfil.columns = dias

fig, ax = plt.subplots(figsize=(8, 3.5))
perfil.plot(ax=ax, linewidth=1.4)
ax.set_xlabel("Hora de llegada")
ax.set_ylabel("Pedidos por hora (media)")
ax.set_title("Demanda media del histórico sintético por hora y día")
ax.legend(ncol=7, fontsize=8, frameon=False)
plt.show()

## 4. Pronóstico

Se compara el ingenuo estacional con la media estacional de las últimas *k* semanas.
*k* se elige en las semanas de validación y el error se informa solo en las semanas de prueba, que no participan de ninguna elección.
La "tasa verdadera" es el error irreducible: lo que queda solo por el azar de las llegadas.

In [ ]:
display(pd.read_csv(RESULTADOS / "pronostico_errores.csv", index_col=["periodo", "metodo"]))
display(pd.read_csv(RESULTADOS / "pronostico_resumen.csv", index_col="indicador").head(5))

semana = pd.read_csv(RESULTADOS / "pronostico_semana_siguiente.csv", index_col="hora", parse_dates=["hora"])
fig, ax = plt.subplots(figsize=(9, 3.2))
x = range(len(semana))
ax.fill_between(x, semana["inferior"], semana["superior"], color="#00798c", alpha=0.2, label="Intervalo empírico")
ax.plot(x, semana["pronostico"], color="#00798c", linewidth=1.4, label="Pronóstico")
inicios = [i for i, h in enumerate(semana.index) if i == 0 or h.normalize() != semana.index[i - 1].normalize()]
ax.set_xticks(inicios, [f"{dias[semana.index[i].day_of_week]} {semana.index[i]:%d/%m}" for i in inicios])
ax.set_ylabel("Pedidos por hora")
ax.set_title("Pronóstico de la semana siguiente al histórico")
ax.legend(frameon=False)
plt.show()

## 5. Verificación del simulador contra M/M/*c*

Con tasas constantes y servicio exponencial, el simulador debe reproducir las fórmulas de Erlang C dentro del intervalo de confianza.
La fila `diferencia_little` comprueba la ley de Little (*L* = λ*W*): su valor teórico es cero.
Esta tabla es la que usa el informe.

In [ ]:
pd.read_csv(RESULTADOS / "verificacion_mmc.csv", index_col="indicador")

### Calculadora M/M/*c* (exploración)

Cambiar λ (pedidos/h), μ (pedidos/h por preparador) y *c*, y ejecutar la celda.
Se comparan las fórmulas de Erlang C con la misma verificación por simulación que usa el informe (unos segundos).
Con ρ = λ/(*c*μ) ≥ 1 no hay régimen estacionario y las fórmulas no aplican.

In [ ]:
LLEGADAS_H = CONFIG["verificacion"]["llegadas_h"]  # λ, pedidos por hora
SERVICIO_H = CONFIG["verificacion"]["servicio_h"]  # μ, pedidos por hora por preparador
DOTACION = CONFIG["verificacion"]["dotacion"]      # c, preparadores

parametros = {**CONFIG["verificacion"], "llegadas_h": LLEGADAS_H, "servicio_h": SERVICIO_H, "dotacion": DOTACION}
print(f"ρ = {LLEGADAS_H / (DOTACION * SERVICIO_H):.3f}")
_, tabla_mmc = verificar_mmc(parametros)
tabla_mmc

## 6. Experimentos: dotación por escenario

Cada escenario es un día de la semana pronosticada: baja (domingo), media (jueves) y alta (sábado).
Para cada dotación se simulan las réplicas de la configuración con números aleatorios comunes: todas las dotaciones reciben exactamente los mismos pedidos.
La dotación recomendada es la menor que alcanza el cumplimiento exigido.
Además, la recomendación se contrasta con los pedidos de las semanas reservadas del histórico, que no se usaron para pronosticar.

In [ ]:
cfg_exp = CONFIG["experimentos"]
print(f"Plazo: {cfg_exp['plazo_min']} min · cumplimiento exigido: {cfg_exp['cumplimiento_min']:.0%} · "
      f"réplicas: {cfg_exp['replicas']} · dotaciones: {cfg_exp['dotaciones']}")

recomendacion = pd.read_csv(RESULTADOS / "experimentos_recomendacion.csv")
columnas = [
    "escenario", "fecha", "demanda_pronosticada", "pico_horario", "dotacion_recomendada",
    "proporcion_a_tiempo", "a_tiempo_ic_semi", "jornadas_cumplen", "espera_media_min", "utilizacion",
    "proporcion_a_tiempo_prueba", "espera_media_min_prueba",
]
display(recomendacion[[c for c in columnas if c in recomendacion.columns]])

# Contraste con los pedidos de las semanas reservadas del histórico.
validacion = RESULTADOS / "experimentos_validacion.csv"
if validacion.exists():
    display(pd.read_csv(validacion))

COLORES = {"baja": "#2b5c8f", "media": "#00798c", "alta": "#d1495b"}


def graficar_desempeno(resumen: pd.DataFrame, cumplimiento_min: float, titulo: str) -> None:
    """Cumplimiento, espera y utilización por dotación para cada escenario."""
    fig, ejes = plt.subplots(1, 3, figsize=(11, 3.2))
    for escenario, grupo in resumen.groupby("escenario", sort=False):
        grupo = grupo.sort_values("dotacion")
        color = COLORES.get(escenario, "black")
        ejes[0].plot(grupo["dotacion"], 100 * grupo["proporcion_a_tiempo"], "o-", color=color, label=escenario)
        ejes[1].plot(grupo["dotacion"], grupo["espera_media_min"], "o-", color=color)
        ejes[2].plot(grupo["dotacion"], 100 * grupo["utilizacion"], "o-", color=color)
    ejes[0].axhline(100 * cumplimiento_min, color="0.4", linestyle="--", linewidth=0.9)
    for ax, etiqueta in zip(ejes, ["Pedidos a tiempo (%)", "Espera media Wq (min)", "Utilización ρ (%)"]):
        ax.set_xlabel("Preparadores (c)")
        ax.set_ylabel(etiqueta)
    ejes[0].legend(frameon=False)
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


graficar_desempeno(pd.read_csv(RESULTADOS / "experimentos_escenarios.csv"), cfg_exp["cumplimiento_min"],
                   "Resultados del informe")

## 7. Sensibilidad

Dotación requerida (`c_rec_*`) y cumplimiento que se obtendría manteniendo la dotación base (`a_tiempo_c_base_*`) cuando:

- la demanda se ubica en los extremos del intervalo del pronóstico o se escala ±10–20 %;
- el tiempo medio de preparación cambia entre −20 % y +40 %.

Un `c_rec` vacío indica que ninguna dotación evaluada cumple el objetivo.

In [ ]:
sens = pd.read_csv(RESULTADOS / "sensibilidad_recomendacion.csv")
sens[["eje", "variante"]
     + [f"c_rec_{e}" for e in ("baja", "media", "alta")]
     + [f"a_tiempo_c_base_{e}" for e in ("baja", "media", "alta")]]

## 8. Exploración: ¿qué pasa si…?

> **Atención:** lo que se calcule acá no figura en el informe. Las variables parten de los valores de `config/experimento.yaml`; si no se modifican, la tabla coincide con la de la sección 6.

Se trabaja sobre una copia de la configuración, así que el archivo original no cambia.
Con las réplicas de la configuración, la celda tarda unos segundos; con menos réplicas es más rápida, pero el resultado tiene más ruido.

In [ ]:
PLAZO_MIN = CONFIG["experimentos"]["plazo_min"]                # minutos desde la llegada hasta pedido listo
CUMPLIMIENTO_MIN = CONFIG["experimentos"]["cumplimiento_min"]  # proporción exigida de pedidos a tiempo
FACTOR_DEMANDA = 1.0                                           # 1.2 = 20 % más pedidos que el pronóstico
MINUTOS_FIJOS = CONFIG["servicio"]["minutos_fijos"]            # minutos por pedido
MINUTOS_POR_LINEA = CONFIG["servicio"]["minutos_por_linea"]    # minutos por artículo distinto
DOTACIONES = CONFIG["experimentos"]["dotaciones"]
REPLICAS = CONFIG["experimentos"]["replicas"]

config_exploracion = copy.deepcopy(CONFIG)
config_exploracion["experimentos"].update(
    plazo_min=PLAZO_MIN, cumplimiento_min=CUMPLIMIENTO_MIN, dotaciones=DOTACIONES, replicas=REPLICAS
)
config_exploracion["servicio"].update(minutos_fijos=MINUTOS_FIJOS, minutos_por_linea=MINUTOS_POR_LINEA)
demanda = semana * FACTOR_DEMANDA

inicio = time.perf_counter()
_, resumen_exploracion = exp.evaluar_escenarios(demanda, config_exploracion)
print(f"Simulado en {time.perf_counter() - inicio:.1f} s")

display(exp.tabla_recomendaciones(resumen_exploracion, CUMPLIMIENTO_MIN)[
    ["escenario", "demanda_pronosticada", "dotacion_recomendada", "dotacion_evaluada",
     "proporcion_a_tiempo", "espera_media_min", "utilizacion"]
])
graficar_desempeno(resumen_exploracion, CUMPLIMIENTO_MIN, "Exploración (no son resultados del informe)")